# ONNX CPU FP32 vs INT8 Comparison

This notebook compares ONNX Runtime CPU FP32 baseline against CPU INT8 dynamic quantized model.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

fp32 = pd.read_json("../results/onnx_cpu_fp32_stage_breakdown.jsonl", lines=True)
int8 = pd.read_json("../results/onnx_cpu_int8_stage_breakdown.jsonl", lines=True)

df = pd.concat([fp32, int8], ignore_index=True)
df.head()

FileNotFoundError: File ../results/onnx_cpu_int8_stage_breakdown.jsonl does not exist

## Summary Table

In [ ]:
cols = [
    "model_variant",
    "precision",
    "dataset",
    "batch_size",
    "max_length",
    "avg_tokens_per_item",
    "embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec",
    "embedding_items_per_sec",
    "end_to_end_items_per_sec",
    "embedding_latency_ms_p95",
    "validation_passed",
    "reference_cosine_similarity_mean",
    "retrieval_top10_overlap",
    "cpu_model_name",
    "cpu_physical_cores",
    "cpu_logical_cores",
    "cpu_has_avx2",
    "cpu_has_avx512",
]

df[cols].sort_values(["dataset", "max_length", "batch_size", "precision"])

## Speedup Table

In [ ]:
key_cols = ["dataset", "batch_size", "max_length"]

fp32_s = fp32[key_cols + [
    "embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec",
    "embedding_latency_ms_p95",
]].rename(columns={
    "embedding_tokens_per_sec": "fp32_embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec": "fp32_end_to_end_tokens_per_sec",
    "embedding_latency_ms_p95": "fp32_embedding_latency_ms_p95",
})

int8_s = int8[key_cols + [
    "embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec",
    "embedding_latency_ms_p95",
    "validation_passed",
    "reference_cosine_similarity_mean",
    "retrieval_top10_overlap",
]].rename(columns={
    "embedding_tokens_per_sec": "int8_embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec": "int8_end_to_end_tokens_per_sec",
    "embedding_latency_ms_p95": "int8_embedding_latency_ms_p95",
})

cmp_df = fp32_s.merge(int8_s, on=key_cols)

cmp_df["embedding_speedup_int8_vs_fp32"] = (
    cmp_df["int8_embedding_tokens_per_sec"]
    / cmp_df["fp32_embedding_tokens_per_sec"]
)

cmp_df["e2e_speedup_int8_vs_fp32"] = (
    cmp_df["int8_end_to_end_tokens_per_sec"]
    / cmp_df["fp32_end_to_end_tokens_per_sec"]
)

cmp_df["p95_latency_ratio_int8_vs_fp32"] = (
    cmp_df["int8_embedding_latency_ms_p95"]
    / cmp_df["fp32_embedding_latency_ms_p95"]
)

cmp_df

## Plot Speedup

In [ ]:
cmp_df["run"] = (
    cmp_df["dataset"].astype(str)
    + ", bs=" + cmp_df["batch_size"].astype(str)
    + ", len=" + cmp_df["max_length"].astype(str)
)

ax = cmp_df.plot.bar(
    x="run",
    y=["embedding_speedup_int8_vs_fp32", "e2e_speedup_int8_vs_fp32"],
    figsize=(16, 6),
)

ax.axhline(1.0, linestyle="--")
ax.axhline(1.10, linestyle=":", color="green", label="+10% speedup threshold")
ax.set_title("ONNX CPU INT8 speedup vs FP32")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("speedup ratio")
ax.legend()
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Plot Validation Quality

In [ ]:
quality_cols = [
    "reference_cosine_similarity_mean",
    "retrieval_top10_overlap",
]

ax = cmp_df.plot.bar(
    x="run",
    y=quality_cols,
    figsize=(16, 6),
)

ax.axhline(0.995, linestyle="--", label="512: cosine >= 0.995")
ax.axhline(0.98, linestyle=":", label="512: top10 >= 0.98")
ax.axhline(0.993, linestyle="--", alpha=0.5, label="128: cosine >= 0.993")
ax.axhline(0.97, linestyle=":", alpha=0.5, label="128: top10 >= 0.97")
ax.axhline(0.990, linestyle="--", alpha=0.3, label="32: cosine >= 0.990")
ax.axhline(0.95, linestyle=":", alpha=0.3, label="32: top10 >= 0.95")
ax.set_title("ONNX CPU INT8 validation quality vs FP32")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("score")
ax.legend()
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Filter Failed INT8 Runs

In [ ]:
failed_int8 = int8[int8["validation_passed"] != True]

failed_cols = [
    "dataset",
    "batch_size",
    "max_length",
    "validation_passed",
    "validation_errors",
    "reference_cosine_similarity_mean",
    "retrieval_top10_overlap",
    "embedding_nan_count",
    "embedding_inf_count",
]

failed_int8[failed_cols]

## Export Comparison CSV

In [ ]:
out = Path("../results/onnx_cpu_fp32_vs_int8.csv")
cmp_df.to_csv(out, index=False)
out

## Speedup Summary by Configuration

In [ ]:
speedup_summary = cmp_df.groupby(["dataset", "max_length"]).agg({
    "embedding_speedup_int8_vs_fp32": ["mean", "min", "max"],
    "e2e_speedup_int8_vs_fp32": ["mean", "min", "max"],
    "validation_passed": "mean",
}).round(3)
speedup_summary.columns = ["_".join(col) for col in speedup_summary.columns]
speedup_summary = speedup_summary.rename(columns={
    "embedding_speedup_int8_vs_fp32_mean": "emb_speedup_mean",
    "embedding_speedup_int8_vs_fp32_min": "emb_speedup_min",
    "embedding_speedup_int8_vs_fp32_max": "emb_speedup_max",
    "e2e_speedup_int8_vs_fp32_mean": "e2e_speedup_mean",
    "e2e_speedup_int8_vs_fp32_min": "e2e_speedup_min",
    "e2e_speedup_int8_vs_fp32_max": "e2e_speedup_max",
    "validation_passed_mean": "validation_pass_rate",
})
speedup_summary

## Decision Summary

In [ ]:
accepted = cmp_df[
    (cmp_df["embedding_speedup_int8_vs_fp32"] > 1.10)
    & (cmp_df["validation_passed"] == True)
]

not_faster = cmp_df[
    (cmp_df["embedding_speedup_int8_vs_fp32"] <= 1.10)
    & (cmp_df["validation_passed"] == True)
]

not_accurate = cmp_df[
    (cmp_df["embedding_speedup_int8_vs_fp32"] > 1.10)
    & (cmp_df["validation_passed"] != True)
]

both_issues = cmp_df[
    (cmp_df["embedding_speedup_int8_vs_fp32"] <= 1.10)
    & (cmp_df["validation_passed"] != True)
]

print(f"INT8 Accepted (speedup > 1.10 and quality passed): {len(accepted)} runs")
print(f"INT8 Accurate but not faster: {len(not_faster)} runs")
print(f"INT8 Faster but not accurate enough: {len(not_accurate)} runs")
print(f"INT8 Both slow and inaccurate: {len(both_issues)} runs")

if len(accepted) > 0:
    print("\nAccepted configurations:")
    print(accepted[["dataset", "batch_size", "max_length", "embedding_speedup_int8_vs_fp32", "reference_cosine_similarity_mean", "retrieval_top10_overlap"]])